<a href="https://colab.research.google.com/github/manulzweb/manulz/blob/master/RedNeuronalSigmoide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
class Neuron:
  def __init__(self, input_size):
    self.pesos = np.random.randn(input_size)
    self.sesgo = np.random.randn()
    self.output = 0
    self.inputs = None
    self.dpeso = np.zeros_like(self.pesos)
    self.dsesgo = 0

  def activate_sigmoide(self, x):
    return 1/(1+np.exp(-x))

  def dactivate_sigmoide(self, x):
    return x*(1-x)

  def forward(self, inputs):
    self.inputs = inputs
    weighted_sum = np.dot(inputs, self.pesos) + self.sesgo
    self.output = self.activate_sigmoide(weighted_sum)
    return self.output

  def backward(self, d_output, tasa_aprendizaje):
    d_activation = d_output * self.dactivate_sigmoide(self.output)
    self.dpeso = np.dot(self.inputs, d_activation)
    self.dsesgo = d_activation
    d_input = np.dot(d_activation, self.pesos)
    self.pesos -= self.dpeso * tasa_aprendizaje
    self.sesgo -= self.dsesgo * tasa_aprendizaje
    return d_input

if __name__ == "__main__":
  neuron = Neuron(3)
  inputs = np.array([1,2,3])
  output = neuron.forward(inputs)
  print("neuron output: ", output)

neuron output:  0.5444398009415544


In [ ]:
import numpy as np
class Layer:
  def __init__(self, num_neurons, inputs_size):
    self.neurons = [Neuron(inputs_size) for _ in range(num_neurons)]

  def forward(self, inputs):
    return np.array([neuron.forward(inputs) for neuron in self.neurons])

  def backward(self, d_output, tasa_aprendizaje):
    d_inputs = np.zeros(len(self.neurons[0].inputs))
    for i, neuron in enumerate(self.neurons):
      d_inputs += neuron.backward(d_output[i], tasa_aprendizaje)
    return d_inputs
if __name__ == "__main__":
  layer = Layer(3, 4)
  inputs = np.array([1, 8, 5, 6])
  layer_output = layer.forward(inputs)
  print("Layer outputs: ", layer_output)

Layer outputs:  [6.25902222e-03 4.91880968e-01 3.09455500e-06]


In [ ]:
import numpy as np
class RedNeuronal:
  def __init__(self):
    self.layers = []
    self.loss_list = []
  def add_layer(self, num_neurons, inputs_size):
    if not self.layers:
      self.layers.append(Layer(num_neurons, inputs_size))
    else:
      previos_output_size = len(self.layers[-1].neurons)
      self.layers.append(Layer(num_neurons, previos_output_size))

  def forward(self, inputs):
    for layer in self.layers:
      inputs = layer.forward(inputs)
    return inputs

  def backward(self, loss_gradient, tasa_aprendizaje):
    for layer in reversed(self.layers):
      loss_gradient = layer.backward(loss_gradient, tasa_aprendizaje)

  def train(self, x, y, epochs=1000, tasa_aprendizaje=0.1):
    for epoch in range(epochs):
      loss = 0
      for i in range(len(x)):
        output = self.forward(x[i])
        loss += np.mean((y[i] - output) ** 2 )
        loss_gradient = 2 * (output - y[i])
        self.backward(loss_gradient, tasa_aprendizaje)
      loss /= len(x)
      self.loss_list.append(loss)
      if epoch & 100 == 0:
        print(f"Epoch: {epoch}, loss {loss}")

  def predict(self, x):
    predictions = []
    for i in range(len(x)):
      predictions.append(self.forward(x[i]))
    return np.array(predictions)

if __name__ == "__main__":
  x = np.array([[0.5, 0.2, 0.1],
                [0.9, 0.7, 0.3],
                [0.4, 0.5, 0.8]])

  y = np.array([[0.3, 0.6, 0.9]]).T

  nn = RedNeuronal()

  nn.add_layer(num_neurons=3, inputs_size=3)
  nn.add_layer(num_neurons=3, inputs_size=3)
  nn.add_layer(num_neurons=1, inputs_size=4)

  nn.train(x,y, epochs=100, tasa_aprendizaje=0.1)

  predictions = nn.predict(x)
  print(f"Predicción: {predictions}")


Epoch: 0, loss 0.16742322218758635
Epoch: 1, loss 0.16595120262725835
Epoch: 2, loss 0.16441554094221786
Epoch: 3, loss 0.16281342902688592
Epoch: 8, loss 0.15370917168516915
Epoch: 9, loss 0.15165152789097022
Epoch: 10, loss 0.1495098673815761
Epoch: 11, loss 0.14728268367321498
Epoch: 16, loss 0.13485379157387198
Epoch: 17, loss 0.1321192192610097
Epoch: 18, loss 0.12931012025846414
Epoch: 19, loss 0.12643280489741757
Epoch: 24, loss 0.11135626440972117
Epoch: 25, loss 0.10829204377038354
Epoch: 26, loss 0.10524772893063056
Epoch: 27, loss 0.10224075014500511
Predicción: [[0.61074141]
 [0.60835022]
 [0.60885947]]
